In [8]:
from IPython.display import HTML, display

html_bridge = """
<div style="border: 3px solid #666; padding: 15px; border-radius: 8px; text-align: center; background-color: #f8f9fa; max-width: 440px; margin: 0 auto; font-family: Arial, sans-serif;">
    <h3 style="color: #333; margin-top: 0;">Local AI Tracker & Diagnostic Console</h3>
    <div style="position: relative; width: 400px; height: 300px; margin: 0 auto; background: #222; border-radius: 4px;">
        <video id="webcam" autoplay playsinline width="400" height="300" style="position: absolute; left: 0; top: 0; border-radius: 4px; transform: scaleX(-1); display: none;"></video>
        <canvas id="output_canvas" width="400" height="300" style="position: absolute; left: 0; top: 0; transform: scaleX(-1);"></canvas>
    </div>
    <div id="gesture_display" style="font-weight: bold; font-size: 18px; color: #007bff; margin-top: 10px;">Status: System Starting...</div>
    <div id="debug_console" style="margin-top: 10px; padding: 8px; background: #111; color: #00ff00; font-family: monospace; font-size: 11px; text-align: left; max-height: 60px; overflow-y: auto; border-radius: 4px;">
        [System] Awaiting script injection hooks...
    </div>
</div>

<script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js" crossorigin="anonymous"></script>
<script src="https://cdn.jsdelivr.net/npm/@mediapipe/drawing_utils/drawing_utils.js" crossorigin="anonymous"></script>
<script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js" crossorigin="anonymous"></script>

<script>
(function() {
    const consoleBox = document.getElementById('debug_console');
    const displayStatus = document.getElementById('gesture_display');
    
    function logMsg(text) {
        consoleBox.innerHTML = "[OK] " + text;
    }

    setTimeout(async () => {
        if (typeof Hands === 'undefined' || typeof Camera === 'undefined') {
            displayStatus.innerText = "Status: Connection Blocked";
            return;
        }
        
        const video = document.getElementById('webcam');
        const canvas = document.getElementById('output_canvas');
        const ctx = canvas.getContext('2d');
        let lastSentGesture = "";

        try {
            const stream = await navigator.mediaDevices.getUserMedia({ video: true });
            video.srcObject = stream;
            video.style.display = "block";
            logMsg("Webcam access active.");
            
            const hands = new Hands({
                locateFile: (file) => `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${file}`
            });
            
            hands.setOptions({
                maxNumHands: 1,
                modelComplexity: 1,
                minDetectionConfidence: 0.5,
                minTrackingConfidence: 0.5
            });

            // FIXED: New rotation-invariant gesture calculator
            function evaluateGesture(landmarks) {
                const wrist = landmarks[0];
                
                // Helper to check if a finger is extended out away from the wrist
                function isExtended(tipIdx, knuckleIdx) {
                    const distTip = Math.hypot(landmarks[tipIdx].x - wrist.x, landmarks[tipIdx].y - wrist.y);
                    const distKnuckle = Math.hypot(landmarks[knuckleIdx].x - wrist.x, landmarks[knuckleIdx].y - wrist.y);
                    return distTip > distKnuckle * 1.15;
                }
                
                const indexExt = isExtended(8, 5);
                const middleExt = isExtended(12, 9);
                const ringExt = isExtended(16, 13);
                const pinkyExt = isExtended(20, 17);
                
                // Total open fingers (excluding thumb)
                const openFingers = (indexExt?1:0) + (middleExt?1:0) + (ringExt?1:0) + (pinkyExt?1:0);
                
                // Check if the thumb tip is extended past its knuckle base
                const thumbExt = Math.hypot(landmarks[4].x - wrist.x, landmarks[4].y - wrist.y) > 
                                 Math.hypot(landmarks[2].x - wrist.x, landmarks[2].y - wrist.y) * 1.1;

                // 1. STOP: All major fingers curled inward
                if (openFingers <= 1 && !thumbExt) return "STOP";
                
                // 2. BACKWARD: Major fingers curled down, but thumb is extended out
                if (openFingers <= 1 && thumbExt) return "BACKWARD";
                
                // 3. DIRECTIONAL MATRICES: 3 or 4 fingers are open
                if (openFingers >= 3) {
                    const fingertipsX = (landmarks[8].x + landmarks[12].x + landmarks[16].x + landmarks[20].x) / 4;
                    const diffX = fingertipsX - wrist.x;
                    
                    // FIXED: Inverted values to match visual mirror alignment perfectly
                    if (diffX < -0.05) return "RIGHT"; 
                    if (diffX > 0.05) return "LEFT";   
                    return "GO";
                }
                
                return "HAND DETECTED";
            }

            hands.onResults((results) => {
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                ctx.drawImage(results.image, 0, 0, canvas.width, canvas.height);
                let currentGesture = "NO HAND DETECTED";
                
                if (results.multiHandLandmarks && results.multiHandLandmarks.length > 0) {
                    const landmarks = results.multiHandLandmarks[0];
                    drawConnectors(ctx, landmarks, HAND_CONNECTIONS, {color: '#00FF00', lineWidth: 4});
                    drawLandmarks(ctx, landmarks, {color: '#FF0000', lineWidth: 2});
                    currentGesture = evaluateGesture(landmarks);
                }
                
                displayStatus.innerText = "Command: " + currentGesture;
                
                if (currentGesture !== lastSentGesture) {
                    if (window.Jupyter && window.Jupyter.notebook) {
                        window.Jupyter.notebook.kernel.execute("global_robot_command = '" + currentGesture + "'");
                    }
                    lastSentGesture = currentGesture;
                }
            });

            const camera = new Camera(video, {
                onFrame: async () => { await hands.send({image: video}); },
                width: 400,
                height: 300
            });
            await camera.start();
            displayStatus.innerText = "Tracking Framework Operational!";
            
        } catch (err) {
            displayStatus.innerText = "Status: Hardware Error";
        }
    }, 500);
})();
</script>
"""
global_robot_command = "NO HAND DETECTED"
display(HTML(html_bridge))

In [7]:
import time 
import threading 
import math
import rclpy 
from rclpy.node import Node
from sensor_msgs.msg import LaserScan
from geometry_msgs.msg import Twist
from robot_wrapper import Robot 

# Reset thread environment flags safely
try:
    thread_active = False
    time.sleep(0.4)
except NameError:
    pass

global_robot_command = "NO HAND DETECTED"
thread_active = True

# -------------------------------------------------------------
# FIXED: LiDAR Node now accepts the Robot engine directly
# -------------------------------------------------------------
class LidarObserver(Node):
    def __init__(self, robot_instance):
        super().__init__('lidar_observer')
        self.robot = robot_instance # Binds the wrapper to prevent pipeline conflict
        self.subscription = self.create_subscription(LaserScan, '/scan', self.avoiding_obstacles, 10)
        self.safety_latch_engaged = False

    def avoiding_obstacles(self, msg):
        ranges = msg.ranges
        if not ranges: return

        # If the safety system is tripped, continuously force the motors to halt
        if self.safety_latch_engaged:
            self.robot.set_cmd_vel(0.0, 0.0, 0.0)
            return

        # FIXED: Look at a 60-degree wide forward cone (30 indices left, 30 indices right)
        # This prevents the robot from getting tunnel vision
        front_cone = ranges[:30] + ranges[-30:]
        
        # Filter out infinite open space readings and NaN sensor glitches
        valid_distances = [r for r in front_cone if 0.0 < r <= 0.50 and not math.isinf(r) and not math.isnan(r)]
        
        if valid_distances:
            closest_obstacle = min(valid_distances)
            self.get_logger().warn(f"🚨 TARGET DETECTED AT {closest_obstacle:.2f}m! SAFETY LATCH ENGAGED! 🚨")
            self.safety_latch_engaged = True
            
            # Stop the robot using the exact same wrapper channel as your hands
            self.robot.set_cmd_vel(0.0, 0.0, 0.0)

# -------------------------------------------------------------
# Background Thread Core Execution Pipeline
# -------------------------------------------------------------
def background_robot_executor():
    global global_robot_command, thread_active
    
    if not rclpy.ok():
        rclpy.init()
        
    robot = Robot() 
    node = LidarObserver(robot) # Pass the active robot controller inside
    
    # FIXED: Group both nodes into a single executor so they spin together smoothly
    executor = rclpy.executors.SingleThreadedExecutor()
    executor.add_node(node)
    executor.add_node(robot)
    
    last_command = None
    print("🤖 Network Command Listener & LiDAR Arc Guard Fully Active.")

    try:
        while rclpy.ok() and thread_active:
            # Spin both nodes simultaneously for balanced thread processing
            executor.spin_once(timeout_sec=0.01)

            current_cmd = global_robot_command

            # Override hand signals instantly if the LiDAR cone trips the safety latch
            if node.safety_latch_engaged:
                current_cmd = "EMERGENCY_STOP"

            if current_cmd != last_command:
                print(f"Server Executing Motion Matrix -> Command: {current_cmd}")
                
                if current_cmd == "STOP" or current_cmd == "EMERGENCY_STOP":
                    robot.set_cmd_vel(0.0, 0.0, 0.0)
                elif current_cmd == "GO":
                    robot.set_cmd_vel(0.2, 0.0, 0.0)
                elif current_cmd == "BACKWARD":
                    robot.set_cmd_vel(-0.4, 0.0, 0.0)
                elif current_cmd == "LEFT":
                    robot.set_cmd_vel(0.0, 1.2, 1.0)
                elif current_cmd == "RIGHT":
                    robot.set_cmd_vel(0.0, -1.2, 1.0)
                    
                last_command = current_cmd
                
            time.sleep(0.02) 

    except Exception as e:
        print(f"Thread encountered error: {e}")
    finally:
        # Safely detach executor bindings on shutdown
        executor.remove_node(node)
        executor.remove_node(robot)
        node.destroy_node()
        print("🛑 Robot background thread shut down cleanly.")

# Spin up the updated multi-node pipeline
worker_thread = threading.Thread(target=background_robot_executor)
worker_thread.start()

🛑 Robot background thread shut down cleanly.


[WARN] [1784746598.978108304] [rcl.logging_rosout]: Publisher already registered for provided node name. If this is due to multiple nodes with the same name then all logs for that logger name will go out over the existing publisher. As soon as any node with that name is destructed it will unregister the publisher, preventing any further logs for that name from being published on the rosout topic.


🤖 Network Command Listener & LiDAR Arc Guard Fully Active.
Server Executing Motion Matrix -> Command: NO HAND DETECTED
